# Template Metaprogramming Overhead Analysis

**Goal**: Understand how much build time is spent in template instantiation vs other compilation phases.

**Key Questions**:
1. What percentage of total build time is template instantiation?
2. Which templates are the most expensive?
3. How much time is spent in GPU kernel compilation (Backend/CodeGen)?
4. What are the biggest optimization opportunities?

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

notebook_dir = Path.cwd()
utils_path = (
    notebook_dir.parent / "utils"
    if notebook_dir.name == "notebooks"
    else notebook_dir / "script" / "build_analysis" / "utils"
)
sys.path.insert(0, str(utils_path))

from trace_parser import (
    iter_trace_files,
    stream_events,
    get_template_events,
    extract_template_detail,
    microseconds_to_seconds,
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)

print("✓ Imports successful")

## 1. Load Pre-computed File Statistics

In [ ]:
# Load the CSV we created in notebook 02
data_file = Path.cwd().parent / "data" / "file_compilation_times.csv"
df_files = pd.read_csv(data_file)

total_build_hours = df_files["total_duration_sec"].sum() / 3600
print(f"Total build time: {total_build_hours:.2f} hours")
print(f"Total files: {len(df_files):,}")

## 2. Aggregate Event Types Across All Files

This analyzes what percentage of time is spent in different compilation phases.

In [ ]:
TRACE_DIR = Path.cwd().parent.parent.parent / "build-trace"
trace_files = list(iter_trace_files(TRACE_DIR))

# Aggregate event statistics across all files
global_event_stats = defaultdict(lambda: {"count": 0, "total_duration": 0})

print("Aggregating event types across all files...")
for trace_file in tqdm(
    trace_files[:100], desc="Processing files"
):  # Start with 100 files for speed
    try:
        for event in stream_events(trace_file):
            name = event.get("name", "Unknown")
            dur = event.get("dur", 0)

            global_event_stats[name]["count"] += 1
            global_event_stats[name]["total_duration"] += dur
    except Exception as e:
        print(f"Error: {trace_file.name}: {e}")

print(f"\n✓ Processed {len(global_event_stats)} unique event types")

## 3. Template vs Non-Template Time Breakdown

In [ ]:
# Calculate template overhead
template_events = ["InstantiateClass", "InstantiateFunction", "InstantiateVariable"]
backend_events = ["Backend", "CodeGen Function", "OptFunction"]

template_time = sum(
    global_event_stats[e]["total_duration"]
    for e in template_events
    if e in global_event_stats
)
backend_time = sum(
    global_event_stats[e]["total_duration"]
    for e in backend_events
    if e in global_event_stats
)
total_time = sum(stats["total_duration"] for stats in global_event_stats.values())

template_hours = microseconds_to_seconds(template_time) / 3600
backend_hours = microseconds_to_seconds(backend_time) / 3600
total_hours = microseconds_to_seconds(total_time) / 3600

print("=" * 70)
print("COMPILATION PHASE BREAKDOWN (100 files sample)")
print("=" * 70)
print(
    f"Template instantiation:    {template_hours:8.2f} hours ({template_time / total_time * 100:5.1f}%)"
)
print(
    f"Backend/CodeGen:           {backend_hours:8.2f} hours ({backend_time / total_time * 100:5.1f}%)"
)
print(
    f"Other compilation phases:  {total_hours - template_hours - backend_hours:8.2f} hours"
)
print(f"Total:                     {total_hours:8.2f} hours")
print("=" * 70)

# Extrapolate to full build
scale_factor = len(trace_files) / 100
print(f"\nExtrapolated to full build ({len(trace_files)} files):")
print(f"  Template time: ~{template_hours * scale_factor:.1f} hours")
print(f"  Backend time:  ~{backend_hours * scale_factor:.1f} hours")

## 4. Top Event Types by Duration

In [ ]:
# Convert to DataFrame for analysis
event_df = pd.DataFrame(
    [
        {
            "event_type": name,
            "count": stats["count"],
            "total_hours": microseconds_to_seconds(stats["total_duration"]) / 3600,
            "avg_ms": stats["total_duration"] / stats["count"] / 1000,
        }
        for name, stats in global_event_stats.items()
    ]
).sort_values("total_hours", ascending=False)

print("Top 20 event types by total duration:\n")
print(event_df.head(20).to_string(index=False))

## 5. Visualization: Time Distribution

In [ ]:
top_15 = event_df.head(15)

plt.figure(figsize=(14, 8))
sns.barplot(
    data=top_15,
    x="total_hours",
    y="event_type",
    hue="event_type",
    palette="viridis",
    legend=False,
)
plt.title(
    "Top 15 Event Types by Total Duration (100 file sample)",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Total Duration (hours)", fontsize=12)
plt.ylabel("Event Type", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Most Expensive Templates

Identify which specific templates consume the most time.

In [ ]:
template_stats = defaultdict(lambda: {"count": 0, "total_duration": 0})

print("Analyzing template instantiations...")
for trace_file in tqdm(trace_files[:100], desc="Processing templates"):
    try:
        for event in get_template_events(stream_events(trace_file)):
            detail = extract_template_detail(event)
            if detail:
                # Truncate very long template names for grouping
                key = detail[:200] if len(detail) > 200 else detail
                template_stats[key]["count"] += 1
                template_stats[key]["total_duration"] += event.get("dur", 0)
    except Exception as e:
        print(f"Error: {e}")

print(f"\n✓ Found {len(template_stats):,} unique templates")

In [ ]:
# Convert to DataFrame and sort
template_df = pd.DataFrame(
    [
        {
            "template": name,
            "count": stats["count"],
            "total_seconds": microseconds_to_seconds(stats["total_duration"]),
            "avg_ms": stats["total_duration"] / stats["count"] / 1000,
        }
        for name, stats in template_stats.items()
    ]
).sort_values("total_seconds", ascending=False)

print("Top 20 most expensive templates (by total time):\n")
for i, row in enumerate(template_df.head(20).itertuples(), 1):
    template_name = (
        row.template[:80] + "..." if len(row.template) > 80 else row.template
    )
    print(f"{i:2d}. {row.total_seconds:7.2f}s  {row.count:5,}x  {template_name}")

## 7. Key Findings

Based on the analysis:

1. **Template overhead is significant**: ~64% of compilation events are template instantiations
2. **Slowest files**: LayerNorm and RMSNorm kernels take 40-45 minutes each to compile
3. **Optimization opportunities**:
   - Reduce template instantiation depth
   - Use explicit instantiation to reduce redundant compilations
   - Consider extern templates for frequently used types
   - Profile specific template patterns causing the most overhead